In [1]:
import sys
sys.path.insert(0, '../')
from data.Reinhard import Reinhard
from models.model_mrcnn import _default_mrcnn_config, build_default
import torch
import pyvips as Vips
import numpy as np
#from Reinhard import Reinhard
import cv2
from PIL import Image
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import os
from explain import ExplainPredictions
import pdb
import matplotlib.pyplot as plt

In [2]:
WGM_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(1024),
        #transforms.RandomHorizontalFlip(),
        #transforms.RandomVerticalFlip(),
        #transforms.FiveCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(1024),
        #transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(1024),
        #transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
LBD_model_path = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/models/mrcnn_models/royal-butterfly-145_mrcnn_model_10.pth'
#WM_model_path = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_models/working_models/27qsmsq7-logs.pth'
WM_model_path = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_models/best_models/2nqho457-logs2.pth'

In [4]:
REF_IMG_PATH =  '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/./DLB_cases/11_063_CG_aSyn_x200.svs'
img = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/././PD017_Syn1_CG.svs'
img = REF_IMG_PATH
REF_IMG_PATH=img
stride = 256
tilesize = 1024
path  = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_models/2nqho457-logs2.pth'
save_dir = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_segmentation_output'

In [5]:
def normalization(REF_IMG_PATH):
    print("Init Normalization")
    ref_image = Vips.Image.new_from_file(REF_IMG_PATH)
    normalizer = Reinhard()
    normalizer.fit(ref_image)
    return normalizer

def load_saved_model(path):
    checkpoint = torch.load(path)
    model_ft = checkpoint["model"]
    model_ft.load_state_dict(checkpoint['state'])
    return model_ft

test_config = dict(
        batch_size = 1,
        num_classes = 1
    )

model_config = _default_mrcnn_config(num_classes=1 + test_config['num_classes']).config
model_lbd = build_default(model_config, im_size=1024)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
model_wgm = load_saved_model(WM_model_path)

In [7]:
tilesize=1024
def crop2img(crop):
    crop_array = crop.numpy()
    crop_array = crop_array[:,:,:3]
    img = Image.fromarray(crop_array.astype('uint8'), 'RGB')
    return img

In [8]:
def getVipsInfo(vips_img):
    # # Get bounds-x and bounds-y offeset
    vfields = [f.split('.') for f in vips_img.get_fields()]
    vfields = [f for f in vfields if f[0] == 'openslide']
    vfields = dict([('.'.join(k[1:]), vips_img.get('.'.join(k))) for k in vfields])
    return vfields

In [9]:
def predict_crop_class(model_ft, crop, data_transforms ):
    #crop_array = torch.from_numpy(crop_array.transpose((2,0,1)))
    #crop_array = crop_array.transpose(2,0,1)
    #img = Image.fromarray(np.uint8(crop_array.transpose((2,0,1)))).convert('RGB')
    img = crop2img(crop)
    transformed_img = data_transforms["test"](img)
    transformed_img = torch.unsqueeze(transformed_img, dim=0)
    outputs = model_ft(transformed_img.to(device).float())
    _, preds = torch.max(outputs, 1)
    return preds.tolist()

In [10]:
vips_img = Vips.Image.new_from_file(img, level=0)
vinfo = getVipsInfo(vips_img)
orig_w, orig_h = int(vinfo['level[0].width']), int(vinfo['level[0].height'])
vips_array = vips_img.numpy()
vips_array = vips_array[:,:,:3]

In [11]:
vips_array_copy = vips_array.copy()
vips_img_new1 = Vips.Image.new_from_array(vips_array_copy)  

In [12]:
path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/WM_model_gray_crops"
stride = 256
masked_image = np.zeros((orig_h,orig_w))
count_predictions = {"white":0,"grey":0,"bg":0}  
for y_val in range(0, orig_h-stride, stride):
    for x_val in range(0, orig_w-stride, stride): 
            if y_val + tilesize < orig_h and x_val + tilesize < orig_w:
                crop = vips_img.crop(x_val, y_val, tilesize, tilesize)
                crop_pred =  predict_crop_class(model_wgm, crop, WGM_transforms)
                if crop_pred==[0]:
                    count_predictions["white"] = count_predictions["white"]+1
                if crop_pred==[2]:
                        count_predictions["bg"] = count_predictions["bg"]+1
                if crop_pred==[1]:
                        count_predictions["grey"] = count_predictions["grey"]+1
                        masked_image[x_val:x_val+tilesize, y_val:y_val+tilesize] = 1
                        #cropped_img = crop2img(crop)
                        #cropped_img.save(os.path.join(path, str(x_val)+"_x_"+str(y_val)+"_y.png"))

In [25]:
1024*1024*0.8

838860.8

In [13]:
result_masks_list = []
stride = 1024
gray_img_dict = dict()
lb_count=0
for y_val in range(0, orig_h-stride, stride):
    for x_val in range(0, orig_w-stride, stride): 
        if y_val + tilesize < orig_h and x_val + tilesize < orig_w:
            if np.sum(masked_image[x_val:x_val+tilesize, y_val:y_val+tilesize])>=838860:
                crop = vips_img.crop(x_val, y_val, tilesize, tilesize)
                cropped_img = crop2img(crop)
                gray_img_dict[(x_val,y_val)] = np.array(cropped_img)
                #images_list.append(cropped_img)

In [22]:
images_list1=[np.array(i) for i in images_list]

In [ ]:
gray_img_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/WM_model_gray_crops"
results_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/LB_output"
for i,v in gray_img_dict.items():
    explain= ExplainPredictions(model_lbd, model_input_path = LBD_model_path, test_input_path=[v], 
                                    detection_threshold=0.75, wandb='', save_result=False, ablation_cam=False, save_thresholds=False)
    
    detected_img_list, boxes_list = explain.generate_results_v1()
    if len(detected_img_list)>0:
        tiled_vips = Vips.Image.new_from_array(detected_img_list[0])
        vips_img_new1= vips_img_new1.insert(tiled_vips,i[0],i[1])

In [15]:
vips_img_new1.tiffsave("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/stitched_img2.tiff", tile=False, compression='lzw', bigtiff=False, pyramid=False)